# Sampled label consistency

This notebook reads the sampled-label export from `data/labels_sampled/`, merges it with the original labels in `data/labels.jsonl`, and measures agreement on the 1-4 execution score using Cohen's weighted kappa.


In [6]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.metrics import cohen_kappa_score

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
LABELS_SAMPLED_DIR = DATA_DIR / "labels_sampled"
LABELS_PATH = DATA_DIR / "labels.jsonl"

pd.set_option("display.max_colwidth", 120)


In [7]:
def extract_first_choice(results, result_type, value_key):
    for result in results:
        if result.get("type") != result_type:
            continue
        value = result.get("value", {}).get(value_key)
        if isinstance(value, list):
            return value[0] if value else None
        return value
    return None


sampled_rows = []
for path in sorted(LABELS_SAMPLED_DIR.iterdir()):
    row = json.loads(path.read_text())
    results = row.get("result", [])
    video_uri = row.get("task", {}).get("data", {}).get("video")
    clip_name = Path(video_uri.split("://", 1)[-1]).name if video_uri else None
    sampled_rows.append(
        {
            "sampled_export_id": row.get("id"),
            "sampled_task_id": row.get("task", {}).get("id"),
            "clip_name": clip_name,
            "sampled_video_uri": video_uri,
            "sampled_created_at": row.get("created_at"),
            "sampled_completed_by": row.get("completed_by", {}).get("email"),
            "sampled_trick": extract_first_choice(results, "choices", "choices"),
            "sampled_score": extract_first_choice(results, "rating", "rating"),
        }
    )

sampled_df = pd.DataFrame(sampled_rows)
sampled_df.head()


,sampled_export_id,sampled_task_id,clip_name,sampled_video_uri,sampled_created_at,sampled_completed_by,sampled_trick,sampled_score
0,1651,10288,25-10-31 19-46-26 5688-00.02.44.697-00.02.49.504-seg09.mp4,s3://aitraf-test/clips/25-10-31 19-46-26 5688-00.02.44.697-00.02.49.504-seg09.mp4,2026-04-20T12:30:25.472229Z,henrikasgir@gmail.com,mizou,4
1,1652,10289,25-10-31 19-46-26 5688-00.04.33.222-00.04.38.667-seg12.mp4,s3://aitraf-test/clips/25-10-31 19-46-26 5688-00.04.33.222-00.04.38.667-seg12.mp4,2026-04-20T12:30:37.324233Z,henrikasgir@gmail.com,None,2
2,1653,10290,25-10-31 19-54-19 5689-00.11.10.367-00.11.15.259-seg20.mp4,s3://aitraf-test/clips/25-10-31 19-54-19 5689-00.11.10.367-00.11.15.259-seg20.mp4,2026-04-20T12:31:54.846706Z,henrikasgir@gmail.com,None,2
3,1654,10291,25-10-31 19-54-19 5689-00.13.09.900-00.13.15.589-seg24.mp4,s3://aitraf-test/clips/25-10-31 19-54-19 5689-00.13.09.900-00.13.15.589-seg24.mp4,2026-04-20T12:32:06.785456Z,henrikasgir@gmail.com,None,2
4,1655,10292,25-11-02 19-14-55 5695-00.00.12.033-00.00.16.478-seg02.mp4,s3://aitraf-test/clips/25-11-02 19-14-55 5695-00.00.12.033-00.00.16.478-seg02.mp4,2026-04-20T12:32:14.989682Z,henrikasgir@gmail.com,None,2


In [8]:
original_df = pd.read_json(LABELS_PATH, lines=True)
original_df["clip_name"] = original_df["video"].map(lambda uri: Path(uri).name)
original_df = original_df.rename(
    columns={
        "video": "original_video_uri",
        "trick": "original_trick",
        "execution_score": "original_score",
        "created_at": "original_created_at",
        "updated_at": "original_updated_at",
        "annotator": "original_annotator",
        "annotation_id": "original_annotation_id",
    }
)

merged_df = sampled_df.merge(
    original_df[
        [
            "clip_name",
            "original_video_uri",
            "original_trick",
            "original_score",
            "original_explanation",
            "original_created_at",
            "original_updated_at",
            "original_annotator",
            "original_annotation_id",
            "person",
            "key_foot",
        ]
    ],
    on="clip_name",
    how="left",
)

merged_df["score_delta"] = merged_df["sampled_score"] - merged_df["original_score"]
merged_df["absolute_score_delta"] = merged_df["score_delta"].abs()
merged_df["exact_match"] = merged_df["sampled_score"] == merged_df["original_score"]

print(f"Sampled export rows: {len(sampled_df)}")
print(f"Matched original labels: {merged_df['original_score'].notna().sum()}")
print(f"Sampled rows with trick labels present: {merged_df['sampled_trick'].notna().sum()}")

merged_df.head()


Sampled export rows: 100
Matched original labels: 100
Sampled rows with trick labels present: 1


,sampled_export_id,sampled_task_id,clip_name,sampled_video_uri,sampled_created_at,sampled_completed_by,sampled_trick,sampled_score,original_video_uri,original_trick,...,original_explanation,original_created_at,original_updated_at,original_annotator,original_annotation_id,person,key_foot,score_delta,absolute_score_delta,exact_match
0,1651,10288,25-10-31 19-46-26 5688-00.02.44.697-00.02.49.504-seg09.mp4,s3://aitraf-test/clips/25-10-31 19-46-26 5688-00.02.44.697-00.02.49.504-seg09.mp4,2026-04-20T12:30:25.472229Z,henrikasgir@gmail.com,mizou,4,s3://aitraf/clips/25-10-31 19-46-26 5688-00.02.44.697-00.02.49.504-seg09.mp4,mizou,...,"Body position on point, but too slow and short",2025-11-09 20:54:12.448141+00:00,2026-01-07 20:20:55.476505+00:00,1,161,Henrikas,right,1,1,False
1,1652,10289,25-10-31 19-46-26 5688-00.04.33.222-00.04.38.667-seg12.mp4,s3://aitraf-test/clips/25-10-31 19-46-26 5688-00.04.33.222-00.04.38.667-seg12.mp4,2026-04-20T12:30:37.324233Z,henrikasgir@gmail.com,None,2,s3://aitraf/clips/25-10-31 19-46-26 5688-00.04.33.222-00.04.38.667-seg12.mp4,top-soul,...,"Position really good, just way too short",2025-11-09 20:54:46.021757+00:00,2026-01-07 14:53:41.775343+00:00,1,164,Max,right,-1,1,False
2,1653,10290,25-10-31 19-54-19 5689-00.11.10.367-00.11.15.259-seg20.mp4,s3://aitraf-test/clips/25-10-31 19-54-19 5689-00.11.10.367-00.11.15.259-seg20.mp4,2026-04-20T12:31:54.846706Z,henrikasgir@gmail.com,None,2,s3://aitraf/clips/25-10-31 19-54-19 5689-00.11.10.367-00.11.15.259-seg20.mp4,ao-soul,...,Very short,2025-11-09 21:00:21.062153+00:00,2026-01-07 20:02:32.656282+00:00,1,190,Max,right,0,0,True
3,1654,10291,25-10-31 19-54-19 5689-00.13.09.900-00.13.15.589-seg24.mp4,s3://aitraf-test/clips/25-10-31 19-54-19 5689-00.13.09.900-00.13.15.589-seg24.mp4,2026-04-20T12:32:06.785456Z,henrikasgir@gmail.com,None,2,s3://aitraf/clips/25-10-31 19-54-19 5689-00.13.09.900-00.13.15.589-seg24.mp4,fs-savanah,...,"Way too short, but form looks confident",2025-11-09 21:01:03.729921+00:00,2026-01-07 20:01:48.924991+00:00,1,192,Henrikas,right,-1,1,False
4,1655,10292,25-11-02 19-14-55 5695-00.00.12.033-00.00.16.478-seg02.mp4,s3://aitraf-test/clips/25-11-02 19-14-55 5695-00.00.12.033-00.00.16.478-seg02.mp4,2026-04-20T12:32:14.989682Z,henrikasgir@gmail.com,None,2,s3://aitraf/clips/25-11-02 19-14-55 5695-00.00.12.033-00.00.16.478-seg02.mp4,fs-royale,...,"Too short, form looks a bit awkward too",2025-11-09 21:01:37.696959+00:00,2026-01-07 20:01:06.403042+00:00,1,194,Romka,left,0,0,True


In [9]:
agreement_df = merged_df.dropna(subset=["sampled_score", "original_score"]).copy()
agreement_df[["sampled_score", "original_score"]] = agreement_df[["sampled_score", "original_score"]].astype(int)

dummy_mode_class = int(agreement_df["original_score"].mode().iloc[0])
agreement_df["dummy_score"] = dummy_mode_class

# Quadratic weights are the primary agreement metric because 1-4 stars are ordinal.
summary_df = pd.DataFrame(
    [
        {"metric": "sampled_export_rows", "value": len(sampled_df)},
        {"metric": "matched_rows", "value": len(agreement_df)},
        {"metric": "dummy_mode_class", "value": dummy_mode_class},
        {"metric": "exact_agreement", "value": float((agreement_df["sampled_score"] == agreement_df["original_score"]).mean())},
        {"metric": "mean_absolute_score_delta", "value": float(agreement_df["absolute_score_delta"].mean())},
        {"metric": "dummy_mae_vs_original", "value": float((agreement_df["original_score"] - agreement_df["dummy_score"]).abs().mean())},
        {"metric": "dummy_mae_vs_sampled", "value": float((agreement_df["sampled_score"] - agreement_df["dummy_score"]).abs().mean())},
        {"metric": "cohen_kappa_unweighted", "value": float(cohen_kappa_score(agreement_df["original_score"], agreement_df["sampled_score"]))},
        {"metric": "cohen_kappa_weighted_linear", "value": float(cohen_kappa_score(agreement_df["original_score"], agreement_df["sampled_score"], weights="linear"))},
        {"metric": "cohen_kappa_weighted_quadratic", "value": float(cohen_kappa_score(agreement_df["original_score"], agreement_df["sampled_score"], weights="quadratic"))},
        {"metric": "dummy_qwk_vs_original", "value": float(cohen_kappa_score(agreement_df["original_score"], agreement_df["dummy_score"], weights="quadratic"))},
        {"metric": "dummy_qwk_vs_sampled", "value": float(cohen_kappa_score(agreement_df["sampled_score"], agreement_df["dummy_score"], weights="quadratic"))},
    ]
)

score_confusion_df = pd.crosstab(
    agreement_df["original_score"],
    agreement_df["sampled_score"],
    rownames=["original_score"],
    colnames=["sampled_score"],
    margins=True,
)

score_delta_df = (
    agreement_df["score_delta"].value_counts().sort_index().rename_axis("score_delta").to_frame("count")
)

score_distribution_df = pd.DataFrame({"score": [1, 2, 3, 4]})
score_distribution_df["original_count"] = score_distribution_df["score"].map(
    agreement_df["original_score"].value_counts()
).fillna(0).astype(int)
score_distribution_df["sampled_count"] = score_distribution_df["score"].map(
    agreement_df["sampled_score"].value_counts()
).fillna(0).astype(int)
score_distribution_df["original_pct"] = (
    score_distribution_df["original_count"] / len(agreement_df)
).round(4)
score_distribution_df["sampled_pct"] = (
    score_distribution_df["sampled_count"] / len(agreement_df)
).round(4)

display(summary_df)
display(score_distribution_df)
display(score_confusion_df)
display(score_delta_df)


,metric,value
0,sampled_export_rows,100.000000
1,matched_rows,100.000000
2,dummy_mode_class,3.000000
3,exact_agreement,0.640000
4,mean_absolute_score_delta,0.370000
5,dummy_mae_vs_original,0.760000
6,dummy_mae_vs_sampled,0.910000
7,cohen_kappa_unweighted,0.513448
8,cohen_kappa_weighted_linear,0.668161
9,cohen_kappa_weighted_quadratic,0.805039


,score,original_count,sampled_count,original_pct,sampled_pct
0,1,13,10,0.13,0.10
1,2,24,39,0.24,0.39
2,3,37,19,0.37,0.19
3,4,26,32,0.26,0.32


sampled_score,1,2,3,4,All
original_score,,,,,
1,9,4,0,0,13
2,1,20,3,0,24
3,0,14,13,10,37
4,0,1,3,22,26
All,10,39,19,32,100


,count
score_delta,
-2,1
-1,18
0,64
1,17


In [10]:
disagreements_df = merged_df[~merged_df["exact_match"]].copy()
disagreements_df = disagreements_df.sort_values(
    ["absolute_score_delta", "clip_name"],
    ascending=[False, True],
)

display(
    disagreements_df[
        [
            "clip_name",
            "person",
            "original_trick",
            "original_score",
            "sampled_score",
            "score_delta",
            "original_explanation",
        ]
    ]
)


,clip_name,person,original_trick,original_score,sampled_score,score_delta,original_explanation
44,25-11-15 19-11-43 5734-00.09.24.041-00.09.29.406-seg13.mp4,Henrikas,bs-royale,4,2,-2,Not perfect but clean enough. Body position should be lower
0,25-10-31 19-46-26 5688-00.02.44.697-00.02.49.504-seg09.mp4,Henrikas,mizou,3,4,1,"Body position on point, but too slow and short"
1,25-10-31 19-46-26 5688-00.04.33.222-00.04.38.667-seg12.mp4,Max,top-soul,3,2,-1,"Position really good, just way too short"
3,25-10-31 19-54-19 5689-00.13.09.900-00.13.15.589-seg24.mp4,Henrikas,fs-savanah,3,2,-1,"Way too short, but form looks confident"
6,25-11-02 19-14-55 5695-00.02.27.610-00.02.32.507-seg11.mp4,Justas,mizou,3,4,1,"A bit too short, but body position on point"
7,25-11-02 19-14-55 5695-00.09.06.529-00.09.10.447-seg36.mp4,Rytis,bs-royale,2,3,1,feet position just not in place in my opinion. Unclear if its a royale or backside
8,25-11-02 19-14-55 5695-00.12.29.692-00.12.33.943-seg61.mp4,Romka,fs-royale,3,4,1,"full box, good speed, sketchy at the end"
10,25-11-07 15-47-00 5699-00.00.39.770-00.00.45.637-seg02.mp4,Max,soul,3,2,-1,"Short, but looks controlled overall"
11,25-11-07 15-47-00 5699-00.00.49.152-00.00.53.874-seg03.mp4,Henrikas,soul,2,3,1,Too short and something about it looks off
13,25-11-07 15-47-00 5699-00.05.10.822-00.05.14.479-seg08.mp4,Max,soul,3,2,-1,"Too short, but great otherwise"
